<a href="https://colab.research.google.com/github/St1CkBSPL/PF174712/blob/main/lab8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install evaluate datasets transformers
import numpy as np
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.metrics import f1_score
from transformers import AutoModelForSequenceClassification, Trainer, TrainingArguments, AutoTokenizer
from datasets import load_dataset
import evaluate


SEED = 67
dataset = load_dataset("imdb")
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, padding="max_length")

tokenized_dataset = dataset.map(tokenize_function, batched=True)

tokenized_dataset = tokenized_dataset.remove_columns(["text"])
tokenized_dataset = tokenized_dataset.rename_column("label", "labels")
tokenized_dataset.set_format("torch")


f1_metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    wynik = f1_metric.compute(predictions=predictions, references=labels, average="macro")
    return {"f1_macro": wynik["f1"]}
rozmiary = [100, 500, 1000, 2000]

wyniki_herbert = {}
wyniki_svm = {}

for n in rozmiary:
    print(f"\n{'='*40}")
    print(f"Trenowanie dla rozmiaru zbioru: {n}")
    print(f"{'='*40}")
    sub_dataset = tokenized_dataset['train'].shuffle(seed=SEED).select(range(n))
    model = AutoModelForSequenceClassification.from_pretrained(
        "distilbert-base-uncased",
        num_labels=2
    )

    training_args = TrainingArguments(
        output_dir=f"./results_rozmiar_{n}",
        num_train_epochs=2,
        learning_rate=2e-5,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        evaluation_strategy="no",
        save_strategy="no",
        report_to="none"
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=sub_dataset,
        eval_dataset=tokenized_dataset['test'],
        compute_metrics=compute_metrics,
    )

    print("Trenowanie DistilBERTa...")
    trainer.train()
    metrics = trainer.evaluate()
    wyniki_herbert[n] = metrics['eval_f1_macro']
    print("Trenowanie klasycznego SVM...")
    X_train_sub = sub_dataset['text']
    y_train_sub = sub_dataset['labels']

    pipe = Pipeline([
        ('tfidf', TfidfVectorizer()),
        ('svm', SVC(kernel='linear', random_state=SEED))
    ])

    pipe.fit(X_train_sub, y_train_sub)

    X_test = tokenized_dataset['test']['text']
    y_test = tokenized_dataset['test']['labels']

    preds_svm = pipe.predict(X_test)
    wyniki_svm[n] = f1_score(y_test, preds_svm, average='macro')

plt.figure(figsize=(10, 6))
x_vals = sorted(rozmiary)
y_herbert = [wyniki_herbert[x] for x in x_vals]
y_svm = [wyniki_svm[x] for x in x_vals]

plt.plot(x_vals, y_herbert, marker='o', linewidth=2, label='DistilBERT (Fine-Tuning)')
plt.plot(x_vals, y_svm, marker='s', linewidth=2, label='TF-IDF + SVM')

plt.title('Wpływ rozmiaru zbioru treningowego: DistilBERT vs SVM (IMDB)')
plt.xlabel('Rozmiar zbioru treningowego (liczba przykładów)')
plt.ylabel('F1 Macro (na zbiorze testowym)')

plt.xscale('log')
plt.xticks(rozmiary, rozmiary)
plt.grid(True, which="both", ls="--", alpha=0.5)
plt.legend()
plt.tight_layout()

plt.show()